# SimCLR Ablation Study — CIFAR-10
**Stage II — Augmentation experiments**

Runs 12 experiments across 3 sessions (split below):
- **Session A** (this notebook): 5 single-augmentation ablations
- **Session B**: 4 pairwise augmentation experiments
- **Session C**: 3 harmful augmentation experiments

Each run: 200 epochs, ResNet-18, batch 256, seed 42, CIFAR-10.  
Expected per-run time: ~1.5–2h on T4. Each session fits in 12h.

**Prerequisites:** The baseline CIFAR-10 screening result must already exist in `linear_eval_results.csv`.

In [ ]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "No GPU."

In [ ]:
import os, subprocess, glob, shutil, csv, datetime

REPO_URL   = "https://github.com/SAlaMusa/IDL.git"
REPO_DIR   = "/kaggle/working/IDL"
OUTPUT_DIR = "/kaggle/working"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)
from models.resnet_simclr import ResNetSimCLR
from optimizers.lars import LARS
print("Imports OK")

## Helper: run one experiment end-to-end

In [ ]:
RESULTS_CSV = os.path.join(OUTPUT_DIR, "ablation_results.csv")

if not os.path.exists(RESULTS_CSV):
    with open(RESULTS_CSV, "w", newline="") as f:
        csv.writer(f).writerow(["exp_name", "config", "best_top1", "checkpoint", "timestamp"])


def run_experiment(exp_name, config_path, epochs=200, batch=256):
    """Pretrain + linear eval for one augmentation experiment.

    Skips pretraining if the checkpoint already exists (crash-safe).
    """
    ckpt_name = f"{exp_name}_ep{epochs}.pth.tar"
    ckpt_path = os.path.join(OUTPUT_DIR, ckpt_name)

    # ── Pretraining
    if os.path.exists(ckpt_path):
        print(f"[{exp_name}] Checkpoint exists, skipping pretraining.")
    else:
        print(f"\n{'='*60}")
        print(f"[{exp_name}] PRETRAINING")
        print(f"{'='*60}")
        result = subprocess.run([
            "python", "run.py",
            "--config", config_path,
            "--epochs",     str(epochs),
            "--batch-size", str(batch),
            "-j", "2", "--seed", "42",
            "--fp16-precision",
        ], capture_output=False)

        if result.returncode != 0:
            raise RuntimeError(f"Pretraining failed for {exp_name}.")

        checkpoints = glob.glob("runs/**/*.pth.tar", recursive=True)
        latest = max(checkpoints, key=os.path.getmtime)
        shutil.copy2(latest, ckpt_path)
        print(f"[{exp_name}] Checkpoint saved: {ckpt_path}")

    # ── Linear Evaluation 
    print(f"\n[{exp_name}] LINEAR EVALUATION")
    eval_result = subprocess.run([
        "python", "linear_eval.py",
        "--checkpoint", ckpt_path,
        "--dataset",    "cifar10",
        "--arch",       "resnet18",
        "--epochs",     "100",
        "-b",           "256",
        "-j",           "2",
        "--seed",       "42",
    ], capture_output=False)

    if eval_result.returncode != 0:
        raise RuntimeError(f"Linear eval failed for {exp_name}.")

    with open("linear_eval_results.csv") as f:
        rows = list(csv.DictReader(f))
    best_top1 = float(rows[-1]["best_top1"])

    with open(RESULTS_CSV, "a", newline="") as f:
        csv.writer(f).writerow([
            exp_name, config_path, f"{best_top1:.2f}", ckpt_path,
            datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
        ])

    print(f"[{exp_name}] Best Top-1: {best_top1:.2f}%")
    return best_top1


print("Helper defined.")

## Session A — Single-Augmentation Ablations
Remove one augmentation at a time. All other augmentations remain as in the baseline.

Expected: ~1.5h × 5 = ~7.5h. Fits in one 12h Kaggle session.

In [ ]:
ablations = [
    ("ablation_no_crop",      "configs/ablation_no_crop.yaml"),
    ("ablation_no_flip",      "configs/ablation_no_flip.yaml"),
    ("ablation_no_jitter",    "configs/ablation_no_jitter.yaml"),
    ("ablation_no_grayscale", "configs/ablation_no_grayscale.yaml"),
    ("ablation_no_blur",      "configs/ablation_no_blur.yaml"),
]

ablation_results = {}
for exp_name, config_path in ablations:
    ablation_results[exp_name] = run_experiment(exp_name, config_path)

print("\nAll ablations complete.")
for name, acc in ablation_results.items():
    print(f"  {name:<30} {acc:.2f}%")

## Session B — Pairwise Augmentation Experiments
Only 2 augmentations active at a time. Run this in a separate Kaggle session.

Expected: ~1.5h × 4 = ~6h.

In [ ]:
pairs = [
    ("pair_crop_jitter",     "configs/pair_crop_jitter.yaml"),
    ("pair_crop_grayscale",  "configs/pair_crop_grayscale.yaml"),
    ("pair_crop_blur",       "configs/pair_crop_blur.yaml"),
    ("pair_jitter_grayscale","configs/pair_jitter_grayscale.yaml"),
]

pair_results = {}
for exp_name, config_path in pairs:
    pair_results[exp_name] = run_experiment(exp_name, config_path)

print("\nAll pairwise experiments complete.")
for name, acc in pair_results.items():
    print(f"  {name:<30} {acc:.2f}%")

## Session C — Harmful Augmentation Experiments
Full baseline + one harmful augmentation added. Run in a separate Kaggle session.

Expected: ~1.5h × 3 = ~4.5h.

In [ ]:
harmful = [
    ("harmful_rotation180", "configs/harmful_rotation180.yaml"),
    ("harmful_noise",       "configs/harmful_noise.yaml"),
    ("harmful_solarize",    "configs/harmful_solarize.yaml"),
]

harmful_results = {}
for exp_name, config_path in harmful:
    harmful_results[exp_name] = run_experiment(exp_name, config_path)

print("\nAll harmful augmentation experiments complete.")
for name, acc in harmful_results.items():
    print(f"  {name:<30} {acc:.2f}%")

## Summary Table
Run this after all three sessions are complete.

In [ ]:
import csv

print(f"{'Experiment':<30} {'Top-1 (%)':>10}")
print("-" * 42)

with open(RESULTS_CSV) as f:
    for row in csv.DictReader(f):
        print(f"{row['exp_name']:<30} {float(row['best_top1']):>10.2f}")

print(f"\nFull results: {RESULTS_CSV}")